# Populate the ontology with concerns and evidence artifacts
Both FairnessConcern and EvidenceArtifact individuals are in the core of the ontology, i.e., into the fairops.ttl file. This notebook loads docs/fairops.ttl, adds FairnessConcerns and EvidenceArtifacts form Fairness_Metrixs.xlsx. Also, the notebook loads docs/indiv.ttl and adds the relations between these FairnessConcerns and EvidenceArtifacts and existing metrics/notions/mitigation techniques in indiv.ttl.
The updated knowledge graph is serialized on oufiles/fairops.ttl and outfile/indiv.ttl. 
If you want changes to be effective, overwrite docs/fairops.ttl and docs/indiv.ttl file with these newly generated files.

In [1]:
from rdflib import Graph
import pandas as pd
from rdflib import URIRef,Literal
from rdflib.namespace import RDF,RDFS,Namespace,OWL,SKOS

In [2]:
# Creating a copy of a given ttl file
g = Graph()
g.parse("./docs/fairops.ttl", format='turtle')
print(f"len(gc)={len(g)}")
CORE = Namespace(str(dict(g.namespaces()).get("core")))
print(SKOS)
#print(g.serialize(format="turtle"))

len(gc)=1141
http://www.w3.org/2004/02/skos/core#


In [3]:
#load the content of the EvidenceArtifacts excel sheet
df = pd.read_excel("../Fairness-Metrics.xlsx", sheet_name="EvidenceArtifacts",header=0, dtype=str)
df.fillna('', inplace=True)

def camelCaseMetric(s):
    temp = s.replace('_', ' ').replace('-', ' ').replace('/', ' ').split('(')[0]
    temp = ' '.join([w.title() if w.islower() else w for w in temp.split()])
    temp=temp.replace(' ', '')
    #res = temp[0].lower() + temp[1:]
    return temp

df['Evidence Artifact Name']=df['Evidence Artifact Name'].apply(lambda x: camelCaseMetric(x))
df.head(2)

,Evidence Artifact Name,Description,Alternative names
0,DatasetRepresentativenessReport,evaluates how well the dataset reflects the ta...,
1,Kolmogorov–SmirnovTest,statistical test used to compare two distribut...,


In [4]:
def addEA(x,definition):
    g.add((CORE[x],RDF.type, OWL.NamedIndividual))
    g.add((CORE[x],RDF.type, CORE.EvidenceArtifact))
    g.add((CORE[x],SKOS.definition,Literal( definition, lang="en") ))

#df.apply(lambda row: print(row),axis=1)
df.apply(lambda row: addEA(row['Evidence Artifact Name'],row['Description']),axis=1)

g.serialize(destination="./outfiles/fairops.ttl")

<Graph identifier=Nc50b6ff396e04898adf9d91a7aec9f17 (<class 'rdflib.graph.Graph'>)>

Now we load the `(Ont) Guidelines` sheet and populate concerns and relations in `indiv.ttl`

In [5]:
df = pd.read_excel("../Fairness-Metrics.xlsx", sheet_name="(Ont) Guidelines",header=0, dtype=str)
df.fillna('', inplace=True)
#df['EvidenceArtifact']=df['EvidenceArtifact'].apply(lambda x: camelCaseMetric(x))


gi = Graph()
gi.parse("./docs/indiv.ttl", format='turtle')
INDIV = Namespace(str(dict(gi.namespaces()).get("indiv")))
print(INDIV)


for index, row in df.iterrows():
    concernIRI=CORE[camelCaseMetric(row['ConcernKeyword'].strip())]
    g.add((concernIRI,SKOS.definition,Literal( row['Definition'], lang="en") ))
    #for articleTriggered in row['Triggered Regulatory req.'].strip().split(',') :
    #    gi.add((concernIRI,CORE.triggers,INDIV[articleTriggered.strip()])) //OBSOLETE RELATION, replaced with the following:    
    for articleTriggered in row['(Triggered) Regulatory req. giving rise to the concern'].strip().split(',') :
        gi.add((INDIV[articleTriggered.strip()],CORE.givesRiseTo,concernIRI))

    for evidArt in row['EvidenceArtifact'].strip().split(',') :
        gi.add((concernIRI,CORE.requires,CORE[camelCaseMetric(evidArt.strip())]))
    
    for notion in row['Notion'].strip().split(',') :
        gi.add((concernIRI,CORE.isAddressedWith,INDIV[camelCaseMetric(notion.strip())]))
    
    #for bias in row['Bias'].strip().split(',') :
    #    gi.add((concernIRI,CORE.exposesTo,CORE[camelCaseMetric(bias.strip())]))
    
g.serialize(destination="./outfiles/fairops.ttl") #adds the fairness concerns definition
gi.serialize(destination="./outfiles/indiv.ttl")

https://purl.org/fairops/indiv#


<Graph identifier=Nda6c78ec61d647998aa97d460916c792 (<class 'rdflib.graph.Graph'>)>